#Project 2
This document has two sections:
###Section 1:
Loading the trained model and performing inference on any of the datasets that the user likes to give

Please follow the instructions given to make predictions.

##Section 2:
This has folder information on validation results from various training runs

###Section 3:
The results of the runs including code for loading training, validation and test data.
The results will be stored in a folder called runs, where each of the validation and test run results are available.




##Common libraries required for running this notebook

In [ ]:
#Install these libraries
!pip install supervision
!pip install ultralytics

In [ ]:
#Necessary Imports
import gdown
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import cv2
import pandas as pd
import os
import random
import json
import albumentations as A
import supervision as sv
import pickle
from ultralytics import YOLO


##Section 1: Inference - Use this to predict using the trained model

In [ ]:
#Download the model parameters and also different run results
url = 'https://drive.google.com/uc?id=1NIhFosSe4ssRRyRkkidwRX-zYkbQWzbh'
output = 'runs.zip'  # Name for the downloaded file
gdown.download(url, output, quiet=False)
!unzip runs.zip

Downloading...
From (original): https://drive.google.com/uc?id=1NIhFosSe4ssRRyRkkidwRX-zYkbQWzbh
From (redirected): https://drive.google.com/uc?id=1NIhFosSe4ssRRyRkkidwRX-zYkbQWzbh&confirm=t&uuid=ce63417a-ae2f-462f-bc64-aaba922742ff
To: /content/runs.zip
100%|██████████| 539M/539M [00:23<00:00, 22.9MB/s]


Archive:  runs.zip
   creating: runs/
   creating: runs/segment/
   creating: runs/segment/val2/
  inflating: runs/segment/val2/BoxPR_curve.png  
  inflating: runs/segment/val2/MaskP_curve.png  
  inflating: runs/segment/val2/val_batch2_pred.jpg  
  inflating: runs/segment/val2/MaskR_curve.png  
  inflating: runs/segment/val2/confusion_matrix_normalized.png  
  inflating: runs/segment/val2/val_batch1_pred.jpg  
  inflating: runs/segment/val2/BoxF1_curve.png  
  inflating: runs/segment/val2/confusion_matrix.png  
  inflating: runs/segment/val2/BoxR_curve.png  
  inflating: runs/segment/val2/val_batch2_labels.jpg  
  inflating: runs/segment/val2/val_batch0_pred.jpg  
  inflating: runs/segment/val2/MaskF1_curve.png  
  inflating: runs/segment/val2/BoxP_curve.png  
  inflating: runs/segment/val2/val_batch0_labels.jpg  
  inflating: runs/segment/val2/MaskPR_curve.png  
  inflating: runs/segment/val2/val_batch1_labels.jpg  
   creating: runs/segment/train/
  inflating: runs/segment/train/Box

In [ ]:
#Downloading test images and labels for inference purposes
testImagesUrl = 'https://drive.google.com/uc?id=1-02BOqkl75yssyd2TJrhkA2T1W9At74r'
testLabelsUrl = 'https://drive.google.com/uc?id=1-3Wy9XtefsCgwHxtBpbxfONKGv7e9iGN'
testYaml = 'https://drive.google.com/uc?id=1IANKFtPeBxGouyZn1QMDFDij5z5eJIoA'
imageOutput = 'testImages.zip'
labelOutput = 'testLabels.zip'
yamlOutput = 'test.yaml'
gdown.download(testImagesUrl, imageOutput, quiet=False)
gdown.download(testLabelsUrl, labelOutput, quiet=False)
gdown.download(testYaml, yamlOutput, quiet=False)
!unzip testImages.zip
!unzip testLabels.zip
!mv ./content/* ./
!rm -r ./content

Downloading...
From (original): https://drive.google.com/uc?id=1-02BOqkl75yssyd2TJrhkA2T1W9At74r
From (redirected): https://drive.google.com/uc?id=1-02BOqkl75yssyd2TJrhkA2T1W9At74r&confirm=t&uuid=9a4db89c-6995-44ee-9da4-c4d7dfbf6529
To: /content/testImages.zip
100%|██████████| 211M/211M [00:01<00:00, 112MB/s]
Downloading...
From: https://drive.google.com/uc?id=1-3Wy9XtefsCgwHxtBpbxfONKGv7e9iGN
To: /content/testLabels.zip
100%|██████████| 52.0k/52.0k [00:00<00:00, 41.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1FRKyhgEnCHcToU2t9ZEnczXlOxP-2uGC
To: /content/test.yaml
100%|██████████| 611/611 [00:00<00:00, 524kB/s]


Archive:  testImages.zip
   creating: content/images/test/
  inflating: content/images/test/Clkv0XmL4Ljo97izdnxfwh1tgZ1cl8cpA0GLZOIU.jpeg  
  inflating: content/images/test/6X1yNVJm9pm5RMXmIybIwIRfYvgDEFebHl4yINEL.jpeg  
  inflating: content/images/test/1xrEl2gcrXSW3F5ZdyUhPGejr8CdB9GjBTEoVW9O.jpeg  
  inflating: content/images/test/iFNAA7JnYGdx9tx315GcZQZUf6dH0E7khiRjkfJO.jpeg  
  inflating: content/images/test/C43svltJ82xrgtT2YlOVTvPtMFFLuiQ4XvwxkRry.jpeg  
  inflating: content/images/test/bLWbA62Iz7UVY4Zudm7kKFIoiWjKOGvDlyNwf9rQ.jpeg  
  inflating: content/images/test/1irxrQqOC1SOvTGQBrNjbnHtL3eHnf6huyHJhqNW.jpeg  
  inflating: content/images/test/OMt9Imf1wcU4fAcLdzwojqxT1mEvFDBbqz2sHZ4O.jpeg  
  inflating: content/images/test/TcZ0wQGfNnPAQPVz5ckhYjQ7Th7DNsWvAA7LOWNt.jpeg  
  inflating: content/images/test/MzBxZFbUdLKntLqG6D91HKorvANYwz4wFHRadRKs.jpeg  
  inflating: content/images/test/aUbWTQHr17Y0HnmIM86Tm1gCNAvDHbMjVAaQIPoE.jpeg  
  inflating: content/images/test/yCXHO4RoXHiibFDRJ

In [ ]:
#Run inference on a set of images
!yolo segment predict model = ./runs/segment/train/weights/best.pt source = ./images/test #change the folder path suitably if needed
#view the results of prediction in './runs/segment/predict

Ultralytics YOLOv8.2.86 🚀 Python-3.10.12 torch-2.4.0+cu121 CUDA:0 (NVIDIA A100-SXM4-40GB, 40514MiB)
YOLOv8m-seg summary (fused): 245 layers, 27,238,596 parameters, 0 gradients, 110.0 GFLOPs

image 1/62 /content/images/test/0AuZ8iMUdcKOVqLFwI4467e4smSjGllWkACcq7jV.jpeg: 1280x960 1 Unlabeled litter, 1 Cigarette, 1 Plastic container, 1 Aluminium foil, 65.0ms
image 2/62 /content/images/test/1Nb1dDfZHtSwykp1r1nx5CFoEE8X8OGcspub9jvf.jpeg: 1280x960 (no detections), 12.3ms
image 3/62 /content/images/test/1QdQ2Oh9jO7161AABdXCyTty8xO3in7K3y9GJWMk.jpeg: 1280x960 (no detections), 12.2ms
image 4/62 /content/images/test/1irxrQqOC1SOvTGQBrNjbnHtL3eHnf6huyHJhqNW.jpeg: 1280x960 1 Paper bag, 12.2ms
image 5/62 /content/images/test/1xrEl2gcrXSW3F5ZdyUhPGejr8CdB9GjBTEoVW9O.jpeg: 960x1280 1 Paper, 1 Aluminium foil, 65.5ms
image 6/62 /content/images/test/2Jj9ssruFI2fnnbND2UOy1PS1x3P2XlL0BpBVyLp.jpeg: 1280x960 1 Paper bag, 1 Carton, 1 Paper, 1 Plastic utensils, 13.0ms
image 7/62 /content/images/test/2Ub46sT99

In [ ]:
#Running validation on images
!yolo segment val model = ./runs/segment/train/weights/best.pt data = ./test.yaml

#Please note.. relative path does not work on Yolo v8 yaml file. The correct absolute path needs to be given to run validation.
#Please fill in the full path of your test images against the 'val:' key in the test.yaml file

Ultralytics YOLOv8.2.86 🚀 Python-3.10.12 torch-2.4.0+cu121 CUDA:0 (NVIDIA A100-SXM4-40GB, 40514MiB)
YOLOv8m-seg summary (fused): 245 layers, 27,238,596 parameters, 0 gradients, 110.0 GFLOPs
val: Scanning /content/labels/test... 62 images, 0 backgrounds, 0 corrupt: 100% 62/62 [00:00<00:00, 774.07it/s]
val: New cache created: /content/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% 4/4 [00:07<00:00,  1.80s/it]
                   all         62        216      0.306      0.288      0.217      0.181      0.297      0.283      0.208      0.175
          Blister pack          5          5      0.318        0.4      0.445      0.429      0.318        0.4      0.445      0.433
                 Straw         12         22      0.112      0.136        0.1     0.0696      0.112      0.136     0.0997     0.0518
      Unlabeled litter          2          2          0          0     0.0146   

##Section 2 - Validation results
After running the above code, you can find a folder called runs/segment
within this folder,
- folder - val - consists of results from the first run of 25 epochs
- folder - val2 - consists of results from the second run of 25 epochs
- folder - val3 - consists of results from the first run of 25 epochs on test images
- folder - val4 - consists of results from the second run of 25 epochs on test images



##Section 3 - Training Information

In [ ]:
import gdown

# URL of the shared file (replace with your own)
url = 'https://drive.google.com/uc?id=195WGReaMJV1ugNYOgIlY46GANg0FsZhH'
output = 'annotations.json'  # Name for the downloaded file
gdown.download(url, output, quiet=False)

url = 'https://drive.google.com/uc?id=17A_ahV3JtvskWMSv3uDabdLDsxl3yR-e'
output = 'test-images-url.csv'
gdown.download(url, output, quiet=False)

url = 'https://drive.google.com/uc?id=1ezwVbP1xPwulqgR9iLPZYkCiuDPWaWs9'
output = 'train-images-url.csv'
gdown.download(url, output, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=195WGReaMJV1ugNYOgIlY46GANg0FsZhH
To: /content/annotations.json
100%|██████████| 3.02M/3.02M [00:00<00:00, 213MB/s]
Downloading...
From: https://drive.google.com/uc?id=17A_ahV3JtvskWMSv3uDabdLDsxl3yR-e
To: /content/test-images-url.csv
100%|██████████| 315k/315k [00:00<00:00, 98.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1ezwVbP1xPwulqgR9iLPZYkCiuDPWaWs9
To: /content/train-images-url.csv
100%|██████████| 242k/242k [00:00<00:00, 81.5MB/s]


'train-images-url.csv'

In [ ]:
f = open('annotations.json')
data = json.load(f)
f.close()
#url = data['images'][0]['flickr_url']
#file_name = os.path.basename(data['images'][0]['file_name'])
#gdown.download(url, os.path.join('/content/', file_name), quiet=True)

###Data Preprocessing functions

In [ ]:
#Set up categories for classification
def setUpCats():
  superCatList = list(set(cat['supercategory'] for cat in data['categories']))
  superCatsDict = dict({i: superCatList[i] for i in range(len(superCatList))})
  return superCatsDict

def catToSuperCatMap(categories):
  superCatDict = setUpCats()
  catToSuperCatDict = {}
  for cat in categories:
    for key in superCatDict.keys():
      if superCatDict[key] == cat['supercategory']:
        catToSuperCatDict[cat['id']] = key
  return catToSuperCatDict

#Albumentation library used for generating altered images data set for training
def transformModel():
  transform = A.Compose([
      A.ColorJitter(p = 0.75),
      A.GaussNoise(p = 0.75),
      A.GaussianBlur(blur_limit=(1,21), sigma_limit=25),
      A.RandomResizedCrop(size = (640, 640), p=0.7),
      A.RandomRotate90(p = 0.5)],
      bbox_params=A.BboxParams(format="coco", label_fields=["bbox_classes"])
  )
  return transform

#Creating folders containing original image dataset
def manageOrigFolders():
  os.system('rm -r images')
  os.system('rm -r labels')
  os.system('mkdir images')
  os.system('mkdir images/train')
  os.system('mkdir images/train/original')
  os.system('mkdir images/test')
  os.system('mkdir images/val')

#Creating folders containing augmented image dataset
def manageAugFolders():
  os.system('rm -r images/train/augment')
  os.system('rm -r labels/train/augment')
  os.system('mkdir images/train/augment')
  os.system('rm -r labels')
  os.system('mkdir labels')
  os.system('mkdir labels/train')
  os.system('mkdir labels/train/original')
  os.system('mkdir labels/train/augment')
  os.system('mkdir labels/test')
  os.system('mkdir labels/val')


#Download the relevant image given an Image ID and Location
def downloadImages(annots, nImages, noDownload):
  testImages = pd.read_csv('test-images-url.csv', names = ('Col1', 'Col2'))
  trainImageList = []
  testImageList = []
  for image in annots['images']:
    if image['id'] < nImages:
      url = image['flickr_url']
      file_name = os.path.basename(image['flickr_url'])#os.path.splitext(os.path.basename(image['flickr_url']))[0]+'.jpg'
      if (url in list(testImages['Col2'])):
        testImageList.append((image['id'], '/content/images/test/'+file_name))
        if not noDownload:
          gdown.download(url, os.path.join('/content/images/test', file_name), quiet=True)
      else:
        trainImageList.append((image['id'], '/content/images/train/original/'+file_name))
        if not noDownload:
          gdown.download(url, os.path.join('/content/images/train/original/', file_name), quiet=True)
  return trainImageList, testImageList

#Convert segmentation information from COCO style to co-ordinate pairs
def convertToPolygons(segList):
  polygons = np.zeros((len(segList))).reshape(-1,2).astype(int)
  for i in range(0, len(segList)-1, 2):
    polygons[int(i/2),0] = segList[i]
    polygons[int(i/2),1] = segList[i+1]
  return polygons

#Convert segmentation to binary marks needed for transformation
def segmentToMasks(image_id, data, width, height):
  polygons = []
  #width, height = data['images'][image_id]['width'], data['images'][image_id]['height']
  #print(f'Width is {width} Height is {height}')
  for annot in data['annotations']:
    if (annot['image_id']==image_id):
      polygons.append(convertToPolygons(annot['segmentation'][0]))
  masks = [ sv.polygon_to_mask(p,(width,height)) for p in polygons ]
  return masks

#Convert Binary masks to segments for the transformed image output
def masksToSegment(masks):
  polygons = []
  for mask in masks:
    # print(mask.shape)
    polygons.append(sv.mask_to_polygons(mask))
  return polygons #[sv.mask_to_polygons(mask) for mask in masks]

#Get the bounding box and classification information from annotations
def getBBoxCats(image_id, data, catToSuperCatDict):
  bbox = []
  cats = []
  for annot in data['annotations']:
    if (annot['image_id'] == image_id):
      bbox.append(annot['bbox'])
      cats.append(catToSuperCatDict[annot['category_id']])
  return bbox, cats
#Given an Image, augment the image and return appropriate masks and categories

#Flattening the masks into segment information needed for COCO style
def flattenList(nestedList):
  flatList = []
  for item in nestedList:
    if (item != []):
       if (len(item) == 1):
         flatList.append(np.ravel(item).tolist())
       else:
         flatList.append(np.ravel(item[0]).tolist())
        # subList = []
        # for subItem in item:
        #    print(subItem)
           #subList.append(np.ravel(subItem).tolist())
        #flatList.append(subList)
  return flatList

#Process to generate augmented images given an Image ID
def augmentImage(annotation, imageId, imgPath,numImage, catToSuperCatDict):
  #Convert the image into a numpy array
   orig_image = np.empty([])
   for image in annotation['images']:
    if (image['id']==imageId):
      pillow_image = Image.open(imgPath)
      orig_image = np.array(pillow_image)
      width, height = orig_image.shape[1],orig_image.shape[0]#image['width'], image['height']
      break
  #Generate a list of masks given the annotations
   masks = segmentToMasks(imageId, annotation, width, height)
  #Get the Boundboxes and the categories given the annotations
   bbox, cats = getBBoxCats(imageId, annotation, catToSuperCatDict)
  #Transform the image
   transform = transformModel()
   directoryStore = '/content/images/train/augment/'
   fileName = str(os.path.splitext(os.path.basename(image['flickr_url']))[0])+'-'+str(numImage) + str(os.path.splitext(os.path.basename(image['flickr_url']))[1])
   try:
      transformed = transform(image=orig_image, masks = masks, bbox_classes = cats) #bboxes = bbox,
      transformed_image = transformed["image"]
      transformed_masks = transformed['masks']
      transformed_cats = transformed['bbox_classes']
      Image.fromarray(transformed_image).save(directoryStore+fileName)
      if (len(transformed_cats) > 0):
        transformed_poly = masksToSegment(transformed_masks)
        flatTransformedPoly = flattenList(transformed_poly)
        #print(transformed_poly)
        width, height = transformed_image.shape[:2]
        segmentToLabel(fileName, flatTransformedPoly, transformed_cats, width, height, True, False)
   except Exception as e:
      print('Transformation Error:' + fileName)
      print(e)
      print(width, height)


#Process to generate Labels in Yolo Format for Train and Test Images
def imageToLabel(image_id, filePath, data, catToSuperCatDict, test=False):
  classes = []
  segments = []
  for annot in data['images']:
    if annot['id'] == image_id:
      width, height = annot['width'], annot['height']
  for annot in data['annotations']:
    if (annot['image_id'] == image_id):
      classes.append(catToSuperCatDict[annot['category_id']])
      segments.append(annot['segmentation'][0])
  fileName = os.path.basename(filePath)
  segmentToLabel(fileName, segments, classes, width, height, test = test)

#Creating YOLO label formats for Train and Test Images
def segmentToLabel(fileName, segList, classes, width, height, augment=False, test=False):
  #Receive
  #create a text file with name and txt in the relevant folder
  if (augment):
    directoryStore = '/content/labels/train/augment/'
  else:
    if test:
      directoryStore = '/content/labels/test/'
    else:
      directoryStore = '/content/labels/train/original/'
  fileName = os.path.splitext(fileName)[0]+'.txt'
  f = open(directoryStore+fileName,'w')
  newSegList = [x for x in segList if x != []]
  for cls, seg in zip(classes, newSegList):
    x_cood = True
    f.write(str(cls)+' ')
    for i in seg:
      f.write(str(i/width)+' ' if x_cood else str(i/height)+' ')
      x_cood = not x_cood
    f.write('\n')
  f.close()

#Function to download training and test images using the annotations file
def downloadMain(nImages, noDownload = False):
  #Read Annotations JSON and process
  f = open('annotations.json')
  data = json.load(f)
  f.close()
  #Download files and create a list
  if not noDownload:
    manageOrigFolders()
  trainImageList, testImageList = downloadImages(data, nImages, noDownload)
  f = open('trainImageList.txt', 'w')
  for image in trainImageList:
    f.write(str(image[0])+','+image[1])
    f.write('\n')
  f.close()
  f = open('testImageList.txt', 'w')
  for image in testImageList:
    f.write(str(image[0])+','+image[1]+'\n')
  f.close()

#Function to generate augmented images and labels
def augmentAndLabel(naugmentImages, testOnly=False):
  catToSuperCatDict = catToSuperCatMap(data['categories'])
  if not testOnly:
    manageAugFolders()
    f = open('trainImageList.txt','r')
    trainImageList = []
    line = f.readline().strip()
    while line !='':
      img_id, imgPath = line.split(",")
      trainImageList.append((int(img_id), imgPath))
      line = f.readline().strip()
    f.close()

    for image in trainImageList:
      imageToLabel(image[0], image[1],data,catToSuperCatDict)
      for i in range(random.randint(1,naugmentImages)): #Generate random number of augmentations
        augmentImage(data, image[0], image[1], i+1, catToSuperCatDict)

  f = open('testImageList.txt','r')
  testImageList = []
  line = f.readline().strip()
  while line !='':
    img_id, imgPath = line.split(",")
    testImageList.append((int(img_id), imgPath))
    line = f.readline().strip()
  f.close()

  for image in testImageList:
    imageToLabel(image[0], image[1],data,catToSuperCatDict,True)

#Function to split original training images to validation set (40%)
def createValSet(p = 0.4):
  f = open('trainImageList.txt','r')
  trainImageList = []
  line = f.readline().strip()
  while line !='':
    img_id, imgPath = line.split(",")
    trainImageList.append((int(img_id), imgPath))
    line = f.readline().strip()
  f.close()

  for image in trainImageList:
    if random.random() < p:
      labelPath = '/content/labels/train/original/'+os.path.splitext(os.path.basename(image[1]))[0]+'.txt'
      os.system('mv '+image[1]+' /content/images/val/')
      os.system('mv '+labelPath+' /content/labels/val/')


In [ ]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

In [ ]:
#-----------------------Download Images--------------------------------
nImages =1500
downloadMain(nImages, noDownload = True)

In [ ]:
#----------------------Augment Images---------------------------------
naugmentImages = 3
augmentAndLabel(naugmentImages, testOnly=True)

In [ ]:
#----------------------Create Validation set-------------------------
createValSet()

In [ ]:
#----------------------Create Yaml file------------------------------
def createYaml(fileType):
  superCatsDict = setUpCats()
  train_image_orig_dir = '/content/images/train/original'
  train_image_aug_dir = '/content/images/train/augment'
  val_image_dir = '/content/images/val'
  test_image_dir = '/content/images/test'

  if fileType == 1:
    fileName = 'data.yaml'
  elif fileType == 2:
    fileName = 'data-base.yaml'
  else:
    fileName = 'test.yaml'
  file = open(fileName,'w')
  file.write('train:\n')
  file.write('  - ' + train_image_orig_dir+'\n')
  if fileType != 2:
    file.write('  - ' + train_image_aug_dir+'\n')
  if fileType == 3:
    file.write('val: ' + test_image_dir +'\n')
  else:
    file.write('val: ' + val_image_dir +'\n')
  file.write('nc: ' + str(len(superCatsDict.keys())) + '\n')
  file.write('names: \n')
  for i in superCatsDict.keys():
    file.write('   ' + str(i) + ': ' + superCatsDict[i] + '\n')
  file.close()

createYaml(1) #data.yaml
createYaml(2) #data-base.yaml
createYaml(3) #test.yaml

##Training

####Baseline

In [ ]:
from ultralytics import YOLO
import torch
model = YOLO("yolov8s-seg.pt")  # load a pretrained model (recommended for training)
results = model.train(data="data-base.yaml", epochs=25, imgsz=1280, save = True, patience = 0, batch = -1)
#/segment/train5

100%|██████████| 22.8M/22.8M [00:00<00:00, 185MB/s]


Ultralytics YOLOv8.2.83 🚀 Python-3.10.12 torch-2.4.0+cu121 CUDA:0 (NVIDIA A100-SXM4-40GB, 40514MiB)
engine/trainer: task=segment, mode=train, model=yolov8s-seg.pt, data=data-base.yaml, epochs=25, time=None, patience=0, batch=-1, imgsz=1280, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train5, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True,

train: Scanning /content/labels/train/original... 879 images, 0 backgrounds, 0 corrupt: 100%|██████████| 879/879 [00:01<00:00, 827.32it/s]

train: New cache created: /content/labels/train/original.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01), CLAHE(p=0.01, clip_limit=(1, 4.0), tile_grid_size=(8, 8))



val: Scanning /content/labels/val.cache... 559 images, 0 backgrounds, 0 corrupt: 100%|██████████| 559/559 [00:00<?, ?it/s]

...

                   all        559       1883      0.429      0.225      0.197      0.159      0.438      0.211       0.19      0.142

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size


      25/25      9.51G     0.6268     0.9885     0.8885     0.9161          9       1280: 100%|██████████| 110/110 [00:26<00:00,  4.21it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 35/35 [00:13<00:00,  2.69it/s]


                   all        559       1883      0.359      0.226      0.196      0.159      0.358      0.223       0.19      0.144

25 epochs completed in 0.275 hours.
Optimizer stripped from runs/segment/train5/weights/last.pt, 23.9MB
Optimizer stripped from runs/segment/train5/weights/best.pt, 23.9MB

Validating runs/segment/train5/weights/best.pt...
Ultralytics YOLOv8.2.83 🚀 Python-3.10.12 torch-2.4.0+cu121 CUDA:0 (NVIDIA A100-SXM4-40GB, 40514MiB)
YOLOv8s-seg summary (fused): 195 layers, 11,790,436 parameters, 0 gradients, 42.5 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 35/35 [00:24<00:00,  1.44it/s]


                   all        559       1883      0.358      0.238      0.197      0.162      0.355      0.236      0.192      0.148
       Styrofoam piece          5          6          1          0    0.00606    0.00606          1          0    0.00606    0.00546
          Blister pack         33         39      0.196      0.333      0.219      0.176      0.196      0.333      0.218      0.172
                 Straw        115        200      0.183      0.144     0.0866     0.0446       0.17      0.134     0.0785     0.0382
      Unlabeled litter         26         29      0.193      0.276      0.158       0.15      0.193      0.276      0.156       0.13
             Cigarette        189        274      0.316      0.511      0.376      0.277      0.316      0.511      0.378      0.267
 Plastic bag & wrapper          3          3          0          0    0.00375    0.00338          0          0    0.00375    0.00375
                   Lid         10         12      0.379     0.0833   

####Improved ones

In [ ]:
from ultralytics import YOLO
import torch
#model = YOLO("yolov8m-seg.pt")  # load a pretrained model (recommended for training)
#segment/train
#second run
model = YOLO("/content/runs/segment/train/weights/last.pt")
results = model.train(data="data.yaml", epochs=25, imgsz=1280, save = True, patience = 0, batch = -1)
#segment/train2


Ultralytics YOLOv8.2.83 🚀 Python-3.10.12 torch-2.4.0+cu121 CUDA:0 (NVIDIA A100-SXM4-40GB, 40514MiB)
engine/trainer: task=segment, mode=train, model=/content/runs/segment/train/weights/last.pt, data=data.yaml, epochs=25, time=None, patience=0, batch=-1, imgsz=1280, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train2, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labe

train: Scanning /content/labels/train/augment.cache... 3764 images, 302 backgrounds, 264 corrupt: 100%|██████████| 3764/3764 [00:00<?, ?it/s]

train: WARNING ⚠️ /content/images/train/augment/0Oh8eforFp1rfrvzIjCqCf4BzqV1688JxGvR1sh8-2.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.2687]
train: WARNING ⚠️ /content/images/train/augment/0bYFDmsNcRP5dyNot5XWTGv1io0Hkufq89h2GT1l-1.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0434      1.2425      1.2421]
train: WARNING ⚠️ /content/images/train/augment/0fb36NSU5KbLHR2pGlVkzb3k5a87u7fzBGemYvNk-3.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0591]
train: WARNING ⚠️ /content/images/train/augment/0k68QlxeXRHkhUOVUAHF8BpZepzsQGUq6rLH42my-1.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.1992]
train: WARNING ⚠️ /content/images/train/augment/0k68QlxeXRHkhUOVUAHF8BpZepzsQGUq6rLH42my-3.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0893]
train: WARNING ⚠️ /content/images/train/augment/0x9


val: Scanning /content/labels/val.cache... 559 images, 0 backgrounds, 0 corrupt: 100%|██████████| 559/559 [00:00<?, ?it/s]

val: WARNING ⚠️ /content/images/val/2FGeAccLObeBVMy8AD0AwyqTDDpWrQlC1ClZ0loI.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/2a7oNJ0IArrvS3z2jub7DaQbsixIhlo32XyRBJEH.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/85utmMwdaTXGtzlZO7E6q8VsGlkHnS7cQDsxw7BO.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/B18ir7xqzn5oHTMBEdm5CghRaJLfYBbINKTWlExg.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/BJDDNwumFrPp7fI43zddmdEzSHlM7rNafD00Wx2Z.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/JslvpcPtajXykA3fz6hs5592F2oqRyZ9cpLYeoas.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/OLA65kyUBC5xdP4ziWzeJVztuBXUtEdEYMRa7aFe.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/WYw6KN5IiVTjyNOTVAraYODJyplxyYaB5dGihl9W.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/XUXe0STwG1ytcyIypo3x0uB1xTfyXFGy40Fi4NK4.jpe

...

                   all        559       1883      0.519      0.317      0.325       0.27      0.513      0.305      0.314      0.237

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size


      25/25      6.92G     0.9111      1.865      1.609      1.249          7       1280: 100%|██████████| 1167/1167 [02:59<00:00,  6.49it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 94/94 [00:11<00:00,  8.15it/s]


                   all        559       1883      0.566      0.317      0.319      0.268      0.539      0.319       0.31      0.237

25 epochs completed in 1.299 hours.
Optimizer stripped from runs/segment/train2/weights/last.pt, 54.9MB
Optimizer stripped from runs/segment/train2/weights/best.pt, 54.9MB

Validating runs/segment/train2/weights/best.pt...
Ultralytics YOLOv8.2.83 🚀 Python-3.10.12 torch-2.4.0+cu121 CUDA:0 (NVIDIA A100-SXM4-40GB, 40514MiB)
YOLOv8m-seg summary (fused): 245 layers, 27,238,596 parameters, 0 gradients, 110.0 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 94/94 [00:09<00:00,  9.55it/s]


                   all        559       1883       0.54      0.303      0.333      0.272      0.529      0.294      0.321      0.239
       Styrofoam piece          5          6      0.632      0.167      0.361      0.298      0.634      0.167      0.361      0.234
          Blister pack         33         39      0.605      0.385      0.422      0.367      0.525      0.333      0.351      0.268
                 Straw        115        200      0.318      0.055      0.131     0.0746      0.322     0.0546      0.128     0.0691
      Unlabeled litter         26         29      0.384       0.31       0.29      0.258      0.385       0.31      0.272      0.211
             Cigarette        189        274      0.406      0.522      0.424      0.333      0.409      0.523      0.424      0.307
 Plastic bag & wrapper          3          3          1          0     0.0574     0.0517          1          0     0.0574     0.0574
                   Lid         10         12      0.531      0.167   

In [ ]:
from ultralytics import YOLO
import torch
#model = YOLO("yolov8m-seg.pt")  # load a pretrained model (recommended for training)
#segment/train
#second run with best param start
model = YOLO("/content/runs/segment/train/weights/best.pt")
results = model.train(data="data.yaml", epochs=25, imgsz=1280, save = True, patience = 0, batch = -1)
#segment/train3

Ultralytics YOLOv8.2.83 🚀 Python-3.10.12 torch-2.4.0+cu121 CUDA:0 (NVIDIA A100-SXM4-40GB, 40514MiB)
engine/trainer: task=segment, mode=train, model=/content/runs/segment/train/weights/best.pt, data=data.yaml, epochs=25, time=None, patience=0, batch=-1, imgsz=1280, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train3, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labe

train: Scanning /content/labels/train/augment.cache... 3764 images, 302 backgrounds, 264 corrupt: 100%|██████████| 3764/3764 [00:00<?, ?it/s]

train: WARNING ⚠️ /content/images/train/augment/0Oh8eforFp1rfrvzIjCqCf4BzqV1688JxGvR1sh8-2.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.2687]
train: WARNING ⚠️ /content/images/train/augment/0bYFDmsNcRP5dyNot5XWTGv1io0Hkufq89h2GT1l-1.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0434      1.2425      1.2421]
train: WARNING ⚠️ /content/images/train/augment/0fb36NSU5KbLHR2pGlVkzb3k5a87u7fzBGemYvNk-3.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0591]
train: WARNING ⚠️ /content/images/train/augment/0k68QlxeXRHkhUOVUAHF8BpZepzsQGUq6rLH42my-1.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.1992]
train: WARNING ⚠️ /content/images/train/augment/0k68QlxeXRHkhUOVUAHF8BpZepzsQGUq6rLH42my-3.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0893]
train: WARNING ⚠️ /content/images/train/augment/0x9


val: Scanning /content/labels/val.cache... 559 images, 0 backgrounds, 0 corrupt: 100%|██████████| 559/559 [00:00<?, ?it/s]

val: WARNING ⚠️ /content/images/val/2FGeAccLObeBVMy8AD0AwyqTDDpWrQlC1ClZ0loI.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/2a7oNJ0IArrvS3z2jub7DaQbsixIhlo32XyRBJEH.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/85utmMwdaTXGtzlZO7E6q8VsGlkHnS7cQDsxw7BO.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/B18ir7xqzn5oHTMBEdm5CghRaJLfYBbINKTWlExg.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/BJDDNwumFrPp7fI43zddmdEzSHlM7rNafD00Wx2Z.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/JslvpcPtajXykA3fz6hs5592F2oqRyZ9cpLYeoas.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/OLA65kyUBC5xdP4ziWzeJVztuBXUtEdEYMRa7aFe.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/WYw6KN5IiVTjyNOTVAraYODJyplxyYaB5dGihl9W.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/XUXe0STwG1ytcyIypo3x0uB1xTfyXFGy40Fi4NK4.jpe

...

                   all        559       1883      0.587      0.342      0.369      0.313      0.579      0.335      0.358      0.274

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size


      25/25      12.2G     0.8156      1.748      1.252      1.158          7       1280: 100%|██████████| 584/584 [02:27<00:00,  3.96it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 47/47 [00:08<00:00,  5.44it/s]


                   all        559       1883      0.578      0.357      0.376       0.32      0.563      0.352      0.365       0.28

25 epochs completed in 1.033 hours.
Optimizer stripped from runs/segment/train3/weights/last.pt, 54.9MB
Optimizer stripped from runs/segment/train3/weights/best.pt, 54.9MB

Validating runs/segment/train3/weights/best.pt...
Ultralytics YOLOv8.2.83 🚀 Python-3.10.12 torch-2.4.0+cu121 CUDA:0 (NVIDIA A100-SXM4-40GB, 40514MiB)
YOLOv8m-seg summary (fused): 245 layers, 27,238,596 parameters, 0 gradients, 110.0 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 47/47 [00:11<00:00,  4.22it/s]


                   all        559       1883      0.542      0.366      0.377      0.318       0.53      0.348      0.361      0.282
       Styrofoam piece          5          6      0.386      0.167      0.324      0.269      0.391      0.167      0.324       0.22
          Blister pack         33         39      0.476      0.615      0.494      0.431       0.47       0.59      0.451      0.348
                 Straw        115        200      0.244      0.185      0.138     0.0837      0.257       0.18      0.136     0.0766
      Unlabeled litter         26         29      0.347      0.414      0.371      0.341      0.338      0.379      0.322      0.293
             Cigarette        189        274       0.43      0.533      0.436      0.342       0.43      0.522      0.428       0.32
 Plastic bag & wrapper          3          3          1          0     0.0135     0.0135          1          0     0.0135     0.0121
                   Lid         10         12      0.778      0.296   

In [ ]:
from ultralytics import YOLO
import torch
#model = YOLO("yolov8m-seg.pt")  # load a pretrained model (recommended for training)
#second run with best param start
model = YOLO("/content/runs/segment/train3/weights/best.pt")
results = model.train(data="data.yaml", epochs=25, imgsz=1280, save = True, patience = 0, batch = -1)
#segment/train4

Ultralytics YOLOv8.2.83 🚀 Python-3.10.12 torch-2.4.0+cu121 CUDA:0 (NVIDIA A100-SXM4-40GB, 40514MiB)
engine/trainer: task=segment, mode=train, model=/content/runs/segment/train3/weights/best.pt, data=data.yaml, epochs=25, time=None, patience=0, batch=-1, imgsz=1280, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train4, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_lab

train: Scanning /content/labels/train/augment.cache... 3764 images, 302 backgrounds, 264 corrupt: 100%|██████████| 3764/3764 [00:00<?, ?it/s]

train: WARNING ⚠️ /content/images/train/augment/0Oh8eforFp1rfrvzIjCqCf4BzqV1688JxGvR1sh8-2.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.2687]
train: WARNING ⚠️ /content/images/train/augment/0bYFDmsNcRP5dyNot5XWTGv1io0Hkufq89h2GT1l-1.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0434      1.2425      1.2421]
train: WARNING ⚠️ /content/images/train/augment/0fb36NSU5KbLHR2pGlVkzb3k5a87u7fzBGemYvNk-3.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0591]
train: WARNING ⚠️ /content/images/train/augment/0k68QlxeXRHkhUOVUAHF8BpZepzsQGUq6rLH42my-1.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.1992]
train: WARNING ⚠️ /content/images/train/augment/0k68QlxeXRHkhUOVUAHF8BpZepzsQGUq6rLH42my-3.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0893]
train: WARNING ⚠️ /content/images/train/augment/0x9


val: Scanning /content/labels/val.cache... 559 images, 0 backgrounds, 0 corrupt: 100%|██████████| 559/559 [00:00<?, ?it/s]

val: WARNING ⚠️ /content/images/val/2FGeAccLObeBVMy8AD0AwyqTDDpWrQlC1ClZ0loI.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/2a7oNJ0IArrvS3z2jub7DaQbsixIhlo32XyRBJEH.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/85utmMwdaTXGtzlZO7E6q8VsGlkHnS7cQDsxw7BO.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/B18ir7xqzn5oHTMBEdm5CghRaJLfYBbINKTWlExg.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/BJDDNwumFrPp7fI43zddmdEzSHlM7rNafD00Wx2Z.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/JslvpcPtajXykA3fz6hs5592F2oqRyZ9cpLYeoas.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/OLA65kyUBC5xdP4ziWzeJVztuBXUtEdEYMRa7aFe.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/WYw6KN5IiVTjyNOTVAraYODJyplxyYaB5dGihl9W.jpeg: corrupt JPEG restored and saved
val: WARNING ⚠️ /content/images/val/XUXe0STwG1ytcyIypo3x0uB1xTfyXFGy40Fi4NK4.jpe

...

                   all        559       1883       0.58      0.341      0.403      0.347      0.565      0.334      0.388      0.306

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size


      25/25      12.2G     0.7428      1.576      0.928      1.101          7       1280: 100%|██████████| 584/584 [02:08<00:00,  4.54it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 47/47 [00:10<00:00,  4.30it/s]


                   all        559       1883      0.554      0.347      0.397      0.339      0.542      0.341      0.385      0.295

25 epochs completed in 1.000 hours.
Optimizer stripped from runs/segment/train4/weights/last.pt, 54.9MB
Optimizer stripped from runs/segment/train4/weights/best.pt, 54.9MB

Validating runs/segment/train4/weights/best.pt...
Ultralytics YOLOv8.2.83 🚀 Python-3.10.12 torch-2.4.0+cu121 CUDA:0 (NVIDIA A100-SXM4-40GB, 40514MiB)
YOLOv8m-seg summary (fused): 245 layers, 27,238,596 parameters, 0 gradients, 110.0 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 47/47 [00:16<00:00,  2.91it/s]


                   all        559       1883      0.578      0.342      0.403      0.347      0.564      0.335      0.388      0.306
       Styrofoam piece          5          6          1      0.413      0.504      0.454          1      0.412      0.504      0.366
          Blister pack         33         39      0.659      0.487      0.547      0.482      0.626      0.462      0.498       0.38
                 Straw        115        200      0.316      0.165      0.145      0.083      0.307       0.16      0.135     0.0718
      Unlabeled litter         26         29      0.419       0.31      0.329      0.322      0.423       0.31      0.331      0.285
             Cigarette        189        274      0.531       0.54      0.507      0.396      0.518      0.526      0.482      0.362
 Plastic bag & wrapper          3          3      0.609      0.333      0.337      0.337       0.61      0.333      0.337      0.337
                   Lid         10         12      0.652      0.333   

###Validation

In [ ]:
#First 25 epochs - Validation set
from ultralytics import YOLO
model = YOLO('runs/segment/train/weights/best.pt')
results_val = model.val(data='data.yaml')
#/content/runs/segment/val

Ultralytics YOLOv8.2.85 🚀 Python-3.10.12 torch-2.4.0+cu121 CUDA:0 (NVIDIA A100-SXM4-40GB, 40514MiB)
YOLOv8m-seg summary (fused): 245 layers, 27,238,596 parameters, 0 gradients, 110.0 GFLOPs


100%|██████████| 755k/755k [00:00<00:00, 14.9MB/s]
val: Scanning /content/labels/val... 559 images, 0 backgrounds, 0 corrupt: 100%|██████████| 559/559 [00:00<00:00, 922.11it/s]


val: New cache created: /content/labels/val.cache


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 35/35 [00:17<00:00,  1.99it/s]


                   all        559       1883      0.549      0.337      0.366      0.308      0.545      0.331      0.352      0.266
       Styrofoam piece          5          6          1      0.232      0.559      0.503          1      0.232      0.559      0.326
          Blister pack         33         39      0.585      0.487       0.45      0.383      0.585      0.487      0.446      0.321
                 Straw        115        200       0.29      0.115       0.13      0.072      0.278       0.11      0.116     0.0572
      Unlabeled litter         26         29      0.411      0.448      0.388      0.368      0.411      0.448      0.387       0.35
             Cigarette        189        274      0.409      0.602      0.471      0.367      0.404      0.595       0.46      0.341
 Plastic bag & wrapper          3          3          1          0      0.337      0.337          1          0      0.337      0.337
                   Lid         10         12       0.51      0.167   

In [ ]:
#Second 25 epochs - Validation set
from ultralytics import YOLO
model = YOLO('runs/segment/train3/weights/best.pt')
results_val = model.val(data='data.yaml')
#/content/runs/segment/val2

Ultralytics YOLOv8.2.85 🚀 Python-3.10.12 torch-2.4.0+cu121 CUDA:0 (NVIDIA A100-SXM4-40GB, 40514MiB)
YOLOv8m-seg summary (fused): 245 layers, 27,238,596 parameters, 0 gradients, 110.0 GFLOPs


val: Scanning /content/labels/val.cache... 559 images, 0 backgrounds, 0 corrupt: 100%|██████████| 559/559 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 35/35 [00:15<00:00,  2.32it/s]


                   all        559       1883      0.549      0.365       0.38      0.322      0.539      0.345      0.363      0.281
       Styrofoam piece          5          6       0.39      0.167      0.325      0.292      0.395      0.167      0.325      0.216
          Blister pack         33         39       0.49      0.615      0.495      0.425      0.492       0.59      0.453      0.339
                 Straw        115        200      0.254      0.177      0.139     0.0838      0.277       0.18      0.134     0.0706
      Unlabeled litter         26         29      0.358      0.414      0.369      0.337      0.328      0.354      0.314      0.283
             Cigarette        189        274      0.435      0.528      0.436      0.344      0.434      0.507      0.431      0.319
 Plastic bag & wrapper          3          3          1          0     0.0143     0.0143          1          0     0.0143     0.0129
                   Lid         10         12      0.761      0.333   

###Test

In [ ]:
#First 25 epochs - Test set
from ultralytics import YOLO
model = YOLO('runs/segment/train/weights/best.pt')
results_test = model.val(data='test.yaml')
#/content/runs/segment/val3

Ultralytics YOLOv8.2.85 🚀 Python-3.10.12 torch-2.4.0+cu121 CUDA:0 (NVIDIA A100-SXM4-40GB, 40514MiB)
YOLOv8m-seg summary (fused): 245 layers, 27,238,596 parameters, 0 gradients, 110.0 GFLOPs


val: Scanning /content/labels/test... 62 images, 0 backgrounds, 0 corrupt: 100%|██████████| 62/62 [00:00<00:00, 833.51it/s]

val: New cache created: /content/labels/test.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:06<00:00,  1.74s/it]


                   all         62        216      0.306      0.288      0.217      0.181      0.297      0.283      0.208      0.175
          Blister pack          5          5      0.318        0.4      0.445      0.429      0.318        0.4      0.445      0.433
                 Straw         12         22      0.112      0.136        0.1     0.0696      0.112      0.136     0.0997     0.0518
      Unlabeled litter          2          2          0          0     0.0146     0.0146          0          0     0.0146     0.0146
             Cigarette         18         31      0.243       0.29      0.139       0.12      0.243       0.29       0.14      0.111
                   Lid          2          4      0.501       0.25      0.251      0.251      0.501       0.25      0.251      0.251
       Plastic glooves          1          1          1          0          0          0          1          0          0          0
             Glass jar         10         12      0.614        0.5   

In [ ]:
#Second 25 epochs - Test set
from ultralytics import YOLO
model = YOLO('runs/segment/train3/weights/best.pt')
results_test = model.val(data='test.yaml')
#/content/runs/segment/val4

Ultralytics YOLOv8.2.85 🚀 Python-3.10.12 torch-2.4.0+cu121 CUDA:0 (NVIDIA A100-SXM4-40GB, 40514MiB)
YOLOv8m-seg summary (fused): 245 layers, 27,238,596 parameters, 0 gradients, 110.0 GFLOPs


val: Scanning /content/labels/test.cache... 62 images, 0 backgrounds, 0 corrupt: 100%|██████████| 62/62 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:07<00:00,  1.80s/it]


                   all         62        216      0.243      0.302      0.209       0.18      0.239      0.293      0.206      0.168
          Blister pack          5          5      0.189        0.4       0.44       0.41      0.191        0.4       0.44       0.38
                 Straw         12         22     0.0822      0.227      0.149      0.104      0.083      0.227       0.15     0.0899
      Unlabeled litter          2          2          0          0     0.0401     0.0401          0          0     0.0401     0.0401
             Cigarette         18         31      0.166      0.258      0.121      0.103      0.167      0.258      0.121     0.0964
                   Lid          2          4      0.403       0.25      0.252      0.252      0.406       0.25      0.252      0.252
       Plastic glooves          1          1          0          0          0          0          0          0          0          0
             Glass jar         10         12      0.405      0.333   

###Housekeeping code - Not to be run

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip '/content/drive/MyDrive/Project/images_aug_val.zip'

Archive:  /content/drive/MyDrive/Project/images_aug_val.zip
   creating: images/
   creating: images/test/
  inflating: images/test/2Jj9ssruFI2fnnbND2UOy1PS1x3P2XlL0BpBVyLp.jpeg  
  inflating: images/test/xKkGejCohal6EZ3DKtPturqNSuimkD9IjUrfj9Iw.jpeg  
  inflating: images/test/wTMf2X4zPFkhKQdYuGRmMobqa6p9qtVuUaq2Qaek.jpeg  
  inflating: images/test/aUbWTQHr17Y0HnmIM86Tm1gCNAvDHbMjVAaQIPoE.jpeg  
  inflating: images/test/nzxNUjAiteVBHI7qp85QdhPYvaiBZIS4g5RvuUA8.jpeg  
  inflating: images/test/dLVNOCFH5mlwf4ITLBVvSdjDRoNTjms4gBtuNqJr.jpeg  
  inflating: images/test/R6LzUiqxF1t3r4VjpCc6Bg62OEygDk8zfGEGpqyd.jpeg  
  inflating: images/test/e9RC2c0jLAXo17Jj5aPQ4EKdivZQS9p7DXV3HTav.jpeg  
  inflating: images/test/r5By2NZUozMsZTexm649uLQ2gk1osmfCbjMTt5lL.jpeg  
  inflating: images/test/TcZ0wQGfNnPAQPVz5ckhYjQ7Th7DNsWvAA7LOWNt.jpeg  
  inflating: images/test/3jW1k8cfLfn8j99NhCWuxTlKsGVLULD9atoASc2n.jpeg  
  inflating: images/test/XHePr7AcpmDs4CIZ23gMl2NiGw4eZQFsqljqRKVB.jpeg  
  inflating: imag

In [ ]:
!unzip '/content/drive/MyDrive/Project/labels_aug_val.zip'
!unzip -j '/content/drive/MyDrive/Project/support_files.zip'

Archive:  /content/drive/MyDrive/Project/labels_aug_val.zip
   creating: labels/
   creating: labels/test/
  inflating: labels/test/1irxrQqOC1SOvTGQBrNjbnHtL3eHnf6huyHJhqNW.txt  
  inflating: labels/test/R6LzUiqxF1t3r4VjpCc6Bg62OEygDk8zfGEGpqyd.txt  
  inflating: labels/test/XHePr7AcpmDs4CIZ23gMl2NiGw4eZQFsqljqRKVB.txt  
  inflating: labels/test/9y9u6Q82zUpFfeUBbmeEQeBfvlbpNxwOjXEWzC1N.txt  
  inflating: labels/test/UrCL0ld6ePFExmgRQhrHnG46posXiwQcoFOGGIV8.txt  
  inflating: labels/test/3jW1k8cfLfn8j99NhCWuxTlKsGVLULD9atoASc2n.txt  
  inflating: labels/test/MYN6gB0AivkEk8Qg676qnsxS4V2nEDHGKW60OSxz.txt  
  inflating: labels/test/TDnMeqRXg7B4sayyCPdFhQWaHfHMsGtubhwUxq55.txt  
  inflating: labels/test/5cFkc7QtoCLrBvfMwwsY5VHbjZBJVic9zk2qlj7W.txt  
  inflating: labels/test/r32Gf0TxYDB69ZUtQwn9cdHsZYmEDUXIY3CpD2VK.txt  
  inflating: labels/test/aUbWTQHr17Y0HnmIM86Tm1gCNAvDHbMjVAaQIPoE.txt  
  inflating: labels/test/nzxNUjAiteVBHI7qp85QdhPYvaiBZIS4g5RvuUA8.txt  
  inflating: labels/test/Jz1j

In [ ]:
!unzip '/content/drive/MyDrive/Project/runs.zip'

In [ ]:
!zip -r testImages.zip /content/images/test
!zip -r testLabels.zip /content/labels/test

  adding: content/images/test/ (stored 0%)
  adding: content/images/test/Clkv0XmL4Ljo97izdnxfwh1tgZ1cl8cpA0GLZOIU.jpeg (deflated 0%)
  adding: content/images/test/6X1yNVJm9pm5RMXmIybIwIRfYvgDEFebHl4yINEL.jpeg (deflated 0%)
  adding: content/images/test/1xrEl2gcrXSW3F5ZdyUhPGejr8CdB9GjBTEoVW9O.jpeg (deflated 0%)
  adding: content/images/test/iFNAA7JnYGdx9tx315GcZQZUf6dH0E7khiRjkfJO.jpeg (deflated 0%)
  adding: content/images/test/C43svltJ82xrgtT2YlOVTvPtMFFLuiQ4XvwxkRry.jpeg (deflated 0%)
  adding: content/images/test/bLWbA62Iz7UVY4Zudm7kKFIoiWjKOGvDlyNwf9rQ.jpeg (deflated 0%)
  adding: content/images/test/1irxrQqOC1SOvTGQBrNjbnHtL3eHnf6huyHJhqNW.jpeg (deflated 0%)
  adding: content/images/test/OMt9Imf1wcU4fAcLdzwojqxT1mEvFDBbqz2sHZ4O.jpeg (deflated 0%)
  adding: content/images/test/TcZ0wQGfNnPAQPVz5ckhYjQ7Th7DNsWvAA7LOWNt.jpeg (deflated 0%)
  adding: content/images/test/MzBxZFbUdLKntLqG6D91HKorvANYwz4wFHRadRKs.jpeg (deflated 0%)
  adding: content/images/test/aUbWTQHr17Y0HnmIM86Tm1gCNAv

In [ ]:
!zip -r /content/drive/MyDrive/Project/valImages.zip /content/images/val
!zip -r /content/drive/MyDrive/Project/valLabels.zip /content/labels/val

  adding: content/images/val/ (stored 0%)
  adding: content/images/val/33978762598_af11912a11_o.png (deflated 0%)
  adding: content/images/val/32912014107_7e5ce18da0_o.png (deflated 0%)
  adding: content/images/val/33979041238_6ae1947d59_o.png (deflated 0%)
  adding: content/images/val/xExRyPGbZa86ntfGsGp72i56mLe6IpqPTagRPsOv.jpeg (deflated 0%)
  adding: content/images/val/46939502215_379cd7fe9b_o.png (deflated 0%)
  adding: content/images/val/2FGeAccLObeBVMy8AD0AwyqTDDpWrQlC1ClZ0loI.jpeg (deflated 1%)
  adding: content/images/val/48661121653_79173c2920_o.png (deflated 0%)
  adding: content/images/val/47066774434_b932933f37_o.png (deflated 0%)
  adding: content/images/val/lfkSqW2Rty5f3ClVo0gpzW3ktQdUeGhftSPjcA6a.jpeg (deflated 0%)
  adding: content/images/val/bmBN1JxNfrmwCpupPg9VspIY8emm8yu3gGCGEgIu.jpeg (deflated 0%)
  adding: content/images/val/48359205447_c02147fb3e_o.png (deflated 0%)
  adding: content/images/val/46939437955_7a28effbcf_o.png (deflated 0%)
  adding: content/images/v

In [ ]:
!zip -r /content/drive/MyDrive/Project/runs-latest.zip runs

  adding: runs/ (stored 0%)
  adding: runs/segment/ (stored 0%)
  adding: runs/segment/val2/ (stored 0%)
  adding: runs/segment/val2/BoxPR_curve.png (deflated 9%)
  adding: runs/segment/val2/MaskP_curve.png (deflated 9%)
  adding: runs/segment/val2/val_batch2_pred.jpg (deflated 1%)
  adding: runs/segment/val2/MaskR_curve.png (deflated 11%)
  adding: runs/segment/val2/confusion_matrix_normalized.png (deflated 11%)
  adding: runs/segment/val2/val_batch1_pred.jpg (deflated 2%)
  adding: runs/segment/val2/BoxF1_curve.png (deflated 11%)
  adding: runs/segment/val2/confusion_matrix.png (deflated 15%)
  adding: runs/segment/val2/BoxR_curve.png (deflated 11%)
  adding: runs/segment/val2/val_batch2_labels.jpg (deflated 1%)
  adding: runs/segment/val2/val_batch0_pred.jpg (deflated 1%)
  adding: runs/segment/val2/MaskF1_curve.png (deflated 11%)
  adding: runs/segment/val2/BoxP_curve.png (deflated 9%)
  adding: runs/segment/val2/val_batch0_labels.jpg (deflated 1%)
  adding: runs/segment/val2/MaskP

In [ ]:
!unzip /content/drive/MyDrive/Project/runs-latest.zip

Archive:  /content/drive/MyDrive/Project/runs-latest.zip
   creating: runs/
   creating: runs/segment/
   creating: runs/segment/val2/
  inflating: runs/segment/val2/BoxPR_curve.png  
  inflating: runs/segment/val2/MaskP_curve.png  
  inflating: runs/segment/val2/val_batch2_pred.jpg  
  inflating: runs/segment/val2/MaskR_curve.png  
  inflating: runs/segment/val2/confusion_matrix_normalized.png  
  inflating: runs/segment/val2/val_batch1_pred.jpg  
  inflating: runs/segment/val2/BoxF1_curve.png  
  inflating: runs/segment/val2/confusion_matrix.png  
  inflating: runs/segment/val2/BoxR_curve.png  
  inflating: runs/segment/val2/val_batch2_labels.jpg  
  inflating: runs/segment/val2/val_batch0_pred.jpg  
  inflating: runs/segment/val2/MaskF1_curve.png  
  inflating: runs/segment/val2/BoxP_curve.png  
  inflating: runs/segment/val2/val_batch0_labels.jpg  
  inflating: runs/segment/val2/MaskPR_curve.png  
  inflating: runs/segment/val2/val_batch1_labels.jpg  
   creating: runs/segment/trai

In [ ]:
!unzip /content/drive/MyDrive/Project/runs-1.zip

In [ ]:
!unzip /content/drive/MyDrive/Project/testImages.zip
!unzip /content/drive/MyDrive/Project/testLabels.zip


Archive:  /content/drive/MyDrive/Project/testImages.zip
   creating: content/images/test/
  inflating: content/images/test/Clkv0XmL4Ljo97izdnxfwh1tgZ1cl8cpA0GLZOIU.jpeg  
  inflating: content/images/test/6X1yNVJm9pm5RMXmIybIwIRfYvgDEFebHl4yINEL.jpeg  
  inflating: content/images/test/1xrEl2gcrXSW3F5ZdyUhPGejr8CdB9GjBTEoVW9O.jpeg  
  inflating: content/images/test/iFNAA7JnYGdx9tx315GcZQZUf6dH0E7khiRjkfJO.jpeg  
  inflating: content/images/test/C43svltJ82xrgtT2YlOVTvPtMFFLuiQ4XvwxkRry.jpeg  
  inflating: content/images/test/bLWbA62Iz7UVY4Zudm7kKFIoiWjKOGvDlyNwf9rQ.jpeg  
  inflating: content/images/test/1irxrQqOC1SOvTGQBrNjbnHtL3eHnf6huyHJhqNW.jpeg  
  inflating: content/images/test/OMt9Imf1wcU4fAcLdzwojqxT1mEvFDBbqz2sHZ4O.jpeg  
  inflating: content/images/test/TcZ0wQGfNnPAQPVz5ckhYjQ7Th7DNsWvAA7LOWNt.jpeg  
  inflating: content/images/test/MzBxZFbUdLKntLqG6D91HKorvANYwz4wFHRadRKs.jpeg  
  inflating: content/images/test/aUbWTQHr17Y0HnmIM86Tm1gCNAvDHbMjVAaQIPoE.jpeg  
  inflating: conten

In [ ]:
!unzip -j '/content/drive/MyDrive/Project/support_files.zip'

Archive:  /content/drive/MyDrive/Project/support_files.zip
  inflating: annotations.json        
  inflating: test-images-url.csv     
  inflating: testImageList.txt       
  inflating: train-images-url.csv    
  inflating: trainImageList.txt      


In [ ]:
!zip -r '/content/drive/MyDrive/Project/images_aug_val.zip' images
!zip -r '/content/drive/MyDrive/Project/labels_aug_val.zip' labels

Streaming output truncated to the last 5000 lines.
  adding: images/val/46939252345_1059f19aa3_o.png (deflated 0%)
  adding: images/val/MwbyFjAhX01Z38d6KthL9bFcJbcXQhGMirJAqUrN.jpeg (deflated 0%)
  adding: images/val/47066870424_f4b18db7c1_o.png (deflated 0%)
  adding: images/val/XC4teyYg9zr5jq505ivm8Z9JmIZp5zLoYOe9ePtW.jpeg (deflated 0%)
  adding: images/val/48661474066_745ae3c4c9_o.png (deflated 0%)
  adding: images/val/40888917783_596fe7cc03_o.png (deflated 0%)
  adding: images/val/47066737454_9c9470d62c_o.png (deflated 0%)
  adding: images/val/VNgmuRkKgVVyIgwJnCITECDEdJdbFD6gFZpxxnlA.jpeg (deflated 0%)
  adding: images/val/47804066862_75470bff8a_o.png (deflated 0%)
  adding: images/val/47066834024_253bdea94a_o.png (deflated 0%)
  adding: images/val/48421438401_8b7495ffa9_o.png (deflated 0%)
  adding: images/val/fGb9UmlFiTAbHvdpvguc4OxZGekzvKKlhLlnBZYb.jpeg (deflated 1%)
  adding: images/val/47803367282_a0c4d52b81_o.png (deflated 0%)
  adding: images/val/48693805828_58336a5377_o.png

In [ ]:
!unzip /content/drive/MyDrive/Project/valImages.zip
!unzip /content/drive/MyDrive/Project/valLabels.zip

Archive:  /content/drive/MyDrive/Project/valImages.zip
   creating: content/images/val/
  inflating: content/images/val/33978762598_af11912a11_o.png  
  inflating: content/images/val/32912014107_7e5ce18da0_o.png  
  inflating: content/images/val/33979041238_6ae1947d59_o.png  
  inflating: content/images/val/xExRyPGbZa86ntfGsGp72i56mLe6IpqPTagRPsOv.jpeg  
  inflating: content/images/val/46939502215_379cd7fe9b_o.png  
  inflating: content/images/val/2FGeAccLObeBVMy8AD0AwyqTDDpWrQlC1ClZ0loI.jpeg  
  inflating: content/images/val/48661121653_79173c2920_o.png  
  inflating: content/images/val/47066774434_b932933f37_o.png  
  inflating: content/images/val/lfkSqW2Rty5f3ClVo0gpzW3ktQdUeGhftSPjcA6a.jpeg  
  inflating: content/images/val/bmBN1JxNfrmwCpupPg9VspIY8emm8yu3gGCGEgIu.jpeg  
  inflating: content/images/val/48359205447_c02147fb3e_o.png  
  inflating: content/images/val/46939437955_7a28effbcf_o.png  
  inflating: content/images/val/46939252345_1059f19aa3_o.png  
  inflating: content/imag

In [ ]:
!unzip '/content/drive/MyDrive/Project/runs.zip'

Archive:  /content/drive/MyDrive/Project/runs.zip
   creating: runs/
   creating: runs/segment/
   creating: runs/segment/train5/
  inflating: runs/segment/train5/val_batch1_pred.jpg  
  inflating: runs/segment/train5/val_batch0_pred.jpg  
  inflating: runs/segment/train5/val_batch1_labels.jpg  
  inflating: runs/segment/train5/BoxR_curve.png  
  inflating: runs/segment/train5/val_batch2_pred.jpg  
  inflating: runs/segment/train5/train_batch1.jpg  
  inflating: runs/segment/train5/BoxF1_curve.png  
  inflating: runs/segment/train5/labels_correlogram.jpg  
  inflating: runs/segment/train5/results.png  
  inflating: runs/segment/train5/val_batch0_labels.jpg  
  inflating: runs/segment/train5/train_batch0.jpg  
  inflating: runs/segment/train5/BoxP_curve.png  
   creating: runs/segment/train5/weights/
  inflating: runs/segment/train5/weights/last.pt  
  inflating: runs/segment/train5/weights/best.pt  
  inflating: runs/segment/train5/results.csv  
  inflating: runs/segment/train5/train_b

In [ ]:
!cp /content/runs/segment/val3/val_*.jpg /content/drive/MyDrive/Project/Images

In [ ]:
!cp ./test.yaml /content/drive/MyDrive/Project